<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Evaluate and use uplift

**[Evaluate and use uplift](https://learning.nextia-ai.com/courses/experiments/m07/evaluate-and-use-uplift/)** · Experiments and Causal Impact: Did Our Change Work? · Module 7, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** draw uplift and Qini curves from a held-out part of the random test, compare three lists of customers, and choose how many customers to email by profit, not by accuracy.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The T-learner and the X-learner, from [Uplift models](https://learning.nextia-ai.com/courses/experiments/m07/uplift-models/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0, matplotlib 3.11.2 and scikit-learn 1.9.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m07/evaluate-and-use-uplift/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M07-L03-evaluate-and-use-uplift/evaluate-and-use-uplift-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, matplotlib 3.11.2 for the charts, and scikit-learn 1.9.1 for the models. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2 scikit-learn==1.9.1
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib", "scikit-learn"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "email/customers.csv.gz": "e16d24a15f1260876a6d73117b24604495f4af0abc855978b7e5b2828ecd3d3c",
    "email/truth.json": "80bc73295fefa15bf0567269825b5e71402abdc98180deaa302e3a05396b9fd3"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Three lists, again

The same split and the same three models as in the previous notebook, in one cell.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression

email = pd.read_csv("data/email/customers.csv.gz")
FEATURES = ["new_customer", "tenure_days", "orders_12m", "spend_12m", "days_since_order", "email_open_rate", "mobile"]
email["mobile"] = (email.device == "mobile").astype(int)
is_test = np.random.default_rng(0).random(len(email)) < 0.3
train, hold = email[~is_test], email[is_test]
X, y, t = train[FEATURES].to_numpy(float), train.ordered.to_numpy(float), (train.variant == "B").to_numpy(int)
Xh = hold[FEATURES].to_numpy(float)
yh, th = hold.ordered.to_numpy(float), (hold.variant == "B").to_numpy(int)

def tree():
    return HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_leaf_nodes=15,
                                         min_samples_leaf=100, random_state=0)

response = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=15,
                                          min_samples_leaf=100, random_state=0).fit(X[t == 1], y[t == 1])
model_a, model_b = tree().fit(X[t == 0], y[t == 0]), tree().fit(X[t == 1], y[t == 1])
effect_b = tree().fit(X[t == 1], y[t == 1] - model_a.predict(X[t == 1]))
effect_a = tree().fit(X[t == 0], model_b.predict(X[t == 0]) - y[t == 0])
share_b = LogisticRegression(max_iter=1000).fit(X, t).predict_proba(Xh)[:, 1]

scores = {"who will buy": response.predict_proba(Xh)[:, 1],
          "T-learner": model_b.predict(Xh) - model_a.predict(Xh),
          "X-learner": share_b * effect_a.predict(Xh) + (1 - share_b) * effect_b.predict(Xh)}
print(len(hold), "customers to evaluate on")

> **Check.** You should see `17843 customers to evaluate on`.

## 2. The uplift curve

Sort the evaluation group by score, best first. Take the top 5%, the top 10%, and so on. In each slice, the extra orders are (share who ordered in B − share who ordered in A) × the number of customers in the slice.

In [ ]:
def uplift_curve(score, y, t, points=20):
    order = np.argsort(-score)
    y, t = y[order], t[order]
    n = len(y)
    shares, gains = [0.0], [0.0]
    for k in np.linspace(n / points, n, points).astype(int):
        yt, yc = y[:k][t[:k] == 1].sum(), y[:k][t[:k] == 0].sum()
        nt, nc = (t[:k] == 1).sum(), (t[:k] == 0).sum()
        gains.append(float((yt / max(nt, 1) - yc / max(nc, 1)) * k))
        shares.append(k / n)
    return np.array(shares), np.array(gains)

curves = {name: uplift_curve(s, yh, th) for name, s in scores.items()}
shares, gains = curves["X-learner"]
print("X-learner, extra orders in the evaluation group:")
for s, g in zip(shares[2::4], gains[2::4]):
    print(f"  top {s:>4.0%}: {g:+7.1f}")

> **Check.** You should see `top  10%:  +176.0` as the first line after the title.

## 3. The Qini coefficient

Emailing at random gives a straight line, from 0 to the extra orders of the whole group. The **Qini coefficient** is the area between the model's curve and that line. Higher is better; 0 means no better than random.

In [ ]:
import matplotlib.pyplot as plt

def qini(shares, gains):
    return float(np.trapezoid(gains - gains[-1] * shares, shares))

for name, (s, g) in curves.items():
    print(f"{name:<14} Qini {qini(s, g):+.2f}")

plt.figure(figsize=(7, 4))
for name, (s, g) in curves.items():
    plt.plot(s * 100, g, marker="o", markersize=3, label=name)
end = curves["X-learner"][1][-1]
plt.plot([0, 100], [0, end], "k--", label="Random")
plt.xlabel("Share of the evaluation group emailed (%)")
plt.ylabel("Extra orders")
plt.legend()
plt.show()

> **Check.** You should see `who will buy   Qini +138.92`, `T-learner      Qini +196.38` and `X-learner      Qini +218.92`: the same Qini coefficients as `xp uplift`.

## 4. From the curve to a budget

Rafael can send 10,000 emails to next month's 50,000 customers: 20% of the list. The evaluation group has fewer customers, so we scale the extra orders of its top 20% up to the 50,000.

In [ ]:
scale = 50_000 / len(hold)
for name, (s, g) in curves.items():
    at20 = np.interp(0.20, s, g) * scale
    print(f"{name:<14} top 20%: about {at20:+.0f} extra orders in 50,000 customers")

> **Check.** You should see `X-learner      top 20%: about +781 extra orders in 50,000 customers` for the X-learner. The simulator's truth for the X-learner's real list is +838: the estimate from 17,843 customers is close, but it is not exact.

## 5. Your turn: profit, not accuracy

Suppose that each extra order earns **26 dollars of margin** and each email costs **0.02 dollars**. Both are assumptions. Profit at a share of the list = extra orders × 26 − emails × 0.02, for the whole list of 50,000 customers.

> **Your turn.** For the X-learner's curve, compute the profit at each share. Put the share with the highest profit in `best_share` (for example `0.55`) and that profit in `best_profit`, rounded to the nearest dollar.

In [ ]:
s, g = curves["X-learner"]
profit = g * scale * 26 - s * 50_000 * 0.02
i = int(np.argmax(profit))
best_share = round(float(s[i]), 2)
best_profit = round(float(profit[i]))
print(best_share, best_profit)

In [ ]:
expect('best_share', best_share, 0.5, tolerance=0.01)

In [ ]:
expect('best_profit', best_profit, 28387, tolerance=30)

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m07/evaluate-and-use-uplift/) compares each list with the simulator's truth, adds the unsubscribes to the profit, and has the knowledge checks that count toward your certificate. Next, Module 8 tests AI features online.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Test a model or a prompt online](https://learning.nextia-ai.com/courses/experiments/m08/test-a-model-or-a-prompt-online/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m07/evaluate-and-use-uplift/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.